
# Algoritmo de Cluster Único de Wolff para o Modelo de Ising 2D

**Material de apoio ao TCC:** *Comparação e Otimização de Algoritmos de Monte Carlo
Aplicados ao Modelo de Ising* (Diego R. Oliveira, UFPR).

Este notebook implementa o algoritmo de **Wolff** (amostragem coletiva não
local, baseada na construção e inversão de um único aglomerado de spins),
seguindo fielmente a formulação apresentada nas Seções **3.6 (Wolff)** e
**4.3.2 (Algoritmo de Cluster Único de Wolff)** do TCC.

O objetivo é que este código sirva tanto para a validação e coleta de dados
do TCC quanto como **material didático para outros alunos da graduação**.

**Estrutura deste notebook:**
1. Probabilidade de ligação (revisão -- já definida em `ising_utils`)
2. Construção do aglomerado (escolha da semente e expansão recursiva)
3. Inversão do aglomerado
4. Um passo completo do algoritmo de Wolff
5. Protocolo completo de simulação (equilibração + produção)
6. Testes de sanidade internos
7. Validação preliminar contra a solução analítica de Onsager

Cada seção de código traz, em comentário, a equação correspondente do TCC.
As funções que descrevem o *sistema físico* (rede, vizinhos, energia,
magnetização, probabilidade de ligação, solução de Onsager) são importadas
de [`ising_utils.py`](https://github.com/diegorafael1010/ising-monte-carlo/blob/main/ising_utils.py),
o mesmo módulo usado em `metropolis.ipynb`. Apenas o que é *específico* do
Wolff (construção do aglomerado, inversão, protocolo de simulação) é
definido diretamente aqui.


In [ ]:

# Clona o repositório do projeto (contém o módulo compartilhado ising_utils.py)
# e adiciona ao caminho de importação do Python.
!git clone https://github.com/diegorafael1010/ising-monte-carlo.git
import sys
sys.path.append('/content/ising-monte-carlo')


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from ising_utils import (
    inicializar_rede,
    construir_tabela_vizinhos,
    energia_total,
    magnetizacao_total,
    probabilidade_ligacao,
    temperatura_critica_onsager,
    energia_onsager,
    magnetizacao_onsager,
    J,
)

# Gerador de números aleatórios com semente fixa, para que os resultados deste
# notebook sejam reprodutíveis. Para rodadas de produção "de verdade", pode-se
# trocar a semente ou deixar aleatória (rng = np.random.default_rng()).
SEED = 42
rng = np.random.default_rng(SEED)



## 1. Probabilidade de Ligação (Revisão)

Conforme demonstrado na Seção 4.3.2 do TCC a partir da condição de balanço
detalhado (Equações 4.10 a 4.15), a probabilidade de ativar uma ligação
entre a semente e um vizinho paralelo é:

$$P_{\text{add}} = 1 - e^{-2\beta J} \qquad \text{(Equações 3.18 / 4.17)}$$

Essa é exatamente a mesma fórmula usada pelo Swendsen-Wang, por isso já
está definida em `ising_utils.py` como `probabilidade_ligacao(beta)` e foi
importada acima. O teste abaixo apenas confirma seu comportamento nos dois
regimes limites.


In [ ]:

# Em T alta (beta -> 0), quase nenhuma ligação deve ser ativada (P_add -> 0).
# Em T baixa (beta -> infinito), quase toda ligação deve ser ativada (P_add -> 1).
print("P_add em T alta  (beta=0.01):", probabilidade_ligacao(beta=0.01))
print("P_add em T baixa (beta=5.0): ", probabilidade_ligacao(beta=5.0))

assert probabilidade_ligacao(beta=0.01) < 0.05
assert probabilidade_ligacao(beta=5.0) > 0.99
print("OK: probabilidade_ligacao se comporta como esperado nos limites de T.")



## 2. Construção do Aglomerado

Seguindo a Seção 4.3.2 do TCC, o aglomerado é construído em quatro etapas:

1. **Escolha da semente**: um sítio $i$ é sorteado uniformemente.
2. **Varredura de vizinhos**: examinam-se os vizinhos do sítio recém-
   adicionado; apenas vizinhos com o **mesmo spin da semente** são
   candidatos a entrar no aglomerado.
3. **Adição estocástica**: cada vizinho candidato entra no aglomerado com
   probabilidade $P_{\text{add}}$.
4. **Expansão recursiva**: o processo se repete para cada novo sítio
   incorporado, até que nenhuma nova ligação seja ativada na fronteira.

Implementamos a expansão de forma **iterativa** (com uma pilha explícita,
`sitios_a_visitar`), em vez de recursiva de fato -- evita o limite de
profundidade de recursão do Python para aglomerados grandes (comuns perto
de $T_c$, onde o aglomerado pode abranger uma fração significativa da rede).
Um array booleano `no_cluster` evita visitar o mesmo sítio duas vezes.


In [ ]:

def construir_cluster(
    estado: np.ndarray,
    vizinhos: np.ndarray,
    p_add: float,
    rng: np.random.Generator,
) -> np.ndarray:
    '''Constrói um aglomerado de Wolff a partir de uma semente aleatória
    (Seção 4.3.2 do TCC).

    Parameters
    ----------
    estado : np.ndarray
        Configuração atual da rede.
    vizinhos : np.ndarray
        Tabela de vizinhos pré-computada (ver ising_utils.construir_tabela_vizinhos).
    p_add : float
        Probabilidade de ligação, P_add = probabilidade_ligacao(beta).
    rng : np.random.Generator
        Gerador de números aleatórios.

    Returns
    -------
    np.ndarray
        Array booleano de tamanho N, com True nos sítios que pertencem ao
        aglomerado construído.
    '''
    N = estado.shape[0]
    no_cluster = np.zeros(N, dtype=bool)

    semente = int(rng.integers(0, N))
    spin_semente = estado[semente]

    no_cluster[semente] = True
    sitios_a_visitar = [semente]

    while sitios_a_visitar:
        sitio_atual = sitios_a_visitar.pop()
        for vizinho in vizinhos[sitio_atual]:
            if no_cluster[vizinho]:
                continue  # já está no aglomerado, não reprocessa
            if estado[vizinho] != spin_semente:
                continue  # spin antiparalelo: probabilidade de adição é nula
            if rng.random() < p_add:
                no_cluster[vizinho] = True
                sitios_a_visitar.append(vizinho)

    return no_cluster


In [ ]:

# Teste de sanidade: em T muito baixa (p_add proximo de 1), o aglomerado a
# partir de uma rede totalmente alinhada (fria) deve cobrir a rede inteira.
L_teste = 16
estado_frio = inicializar_rede(L_teste, modo="fria")
viz_teste = construir_tabela_vizinhos(L_teste)

cluster_grande = construir_cluster(estado_frio, viz_teste, p_add=0.999, rng=rng)
print(f"Fracao da rede no aglomerado (T baixa, rede fria): {cluster_grande.mean():.3f} (esperado: proximo de 1)")
assert cluster_grande.mean() > 0.95

# Em T muito alta (p_add proximo de 0), o aglomerado deve conter só a semente
# (ou quase), mesmo partindo de uma rede fria.
cluster_pequeno = construir_cluster(estado_frio, viz_teste, p_add=0.001, rng=rng)
print(f"Tamanho do aglomerado (T alta, p_add~0): {cluster_pequeno.sum()} sitio(s) (esperado: bem pequeno)")
assert cluster_pequeno.sum() < 5
print("OK: construir_cluster se comporta como esperado nos limites de T.")



## 3. Inversão do Aglomerado

Concluída a construção, **todos** os sítios do aglomerado são invertidos
simultaneamente (Equação 4.16 do TCC):

$$s_m \to -s_m, \quad \forall m \in C$$

Como visto na Seção 4.3.2, essa inversão é sempre aceita com probabilidade 1
(a razão de aceitação $A(\mu\to\nu)=1$ foi garantida pela própria escolha de
$P_{\text{add}}$ via balanço detalhado) -- por isso não há etapa de rejeição
estocástica aqui, diferente do Metropolis.


In [ ]:

def inverter_cluster(estado: np.ndarray, no_cluster: np.ndarray) -> None:
    '''Inverte, in-place, todos os spins pertencentes ao aglomerado
    (Equação 4.16 do TCC).
    '''
    estado[no_cluster] *= -1


In [ ]:

# Teste de sanidade: apos inverter um aglomerado conhecido, exatamente os
# sitios marcados devem trocar de sinal, e nenhum outro.
L_teste = 8
estado_teste = inicializar_rede(L_teste, modo="fria")
viz_teste = construir_tabela_vizinhos(L_teste)

cluster_teste = construir_cluster(estado_teste, viz_teste, p_add=0.5, rng=rng)
estado_antes = estado_teste.copy()

inverter_cluster(estado_teste, cluster_teste)

# Sitios no cluster devem ter trocado de sinal; os demais, permanecido iguais.
assert np.all(estado_teste[cluster_teste] == -estado_antes[cluster_teste])
assert np.all(estado_teste[~cluster_teste] == estado_antes[~cluster_teste])
print(f"OK: inverter_cluster trocou exatamente os {cluster_teste.sum()} sitios do aglomerado.")



## 4. Um Passo Completo do Algoritmo de Wolff

Como discutido na Seção 4.3.2 do TCC, **uma atualização** do algoritmo de
Wolff corresponde à construção e inversão de **um único aglomerado** --
diferente do Metropolis, em que 1 MCS/site equivale a $N=L^2$ tentativas.
Essa diferença de escala temporal é importante e será tratada com cuidado
na análise de eficiência estatística (Seção 4.7 do TCC).

A função abaixo também **retorna o tamanho do aglomerado**, $n_{\text{cluster}}$
-- essa grandeza não é apenas um diagnóstico: pela Equação (3.20) do TCC,
seu valor médio $\langle n \rangle$ está diretamente ligado à susceptibilidade
magnética do sistema (o chamado *estimador melhorado* de Sweeny e Wolff), e
será usada na normalização do tempo de autocorrelação efetivo do Wolff
(Equação 4.47, Seção 4.7.4).


In [ ]:

def passo_wolff(
    estado: np.ndarray,
    vizinhos: np.ndarray,
    beta: float,
    rng: np.random.Generator,
) -> int:
    '''Executa uma atualização completa do algoritmo de Wolff: constrói um
    aglomerado e o inverte (Seção 4.3.2 do TCC).

    Returns
    -------
    int
        Tamanho do aglomerado invertido (n_cluster), usado tanto como
        diagnóstico quanto como insumo para o estimador melhorado da
        susceptibilidade (Equação 3.20) e para a normalização do tempo de
        autocorrelação efetivo do Wolff (Equação 4.47).
    '''
    p_add = probabilidade_ligacao(beta)
    no_cluster = construir_cluster(estado, vizinhos, p_add, rng)
    inverter_cluster(estado, no_cluster)
    return int(no_cluster.sum())


In [ ]:

# Teste rapido: alguns passos de Wolff em T alta devem produzir aglomerados
# pequenos (poucos spins invertidos por passo); em T baixa, aglomerados grandes.
L_teste = 32
viz_teste = construir_tabela_vizinhos(L_teste)

estado_T_alta = inicializar_rede(L_teste, modo="quente", rng=rng)
tamanhos_T_alta = [passo_wolff(estado_T_alta, viz_teste, beta=0.1, rng=rng) for _ in range(20)]

estado_T_baixa = inicializar_rede(L_teste, modo="fria", rng=rng)
tamanhos_T_baixa = [passo_wolff(estado_T_baixa, viz_teste, beta=1.5, rng=rng) for _ in range(20)]

print(f"Tamanho medio do cluster em T alta (beta=0.1): {np.mean(tamanhos_T_alta):.1f} sitios")
print(f"Tamanho medio do cluster em T baixa (beta=1.5): {np.mean(tamanhos_T_baixa):.1f} sitios")
assert np.mean(tamanhos_T_alta) < np.mean(tamanhos_T_baixa)
print("OK: aglomerados sao maiores em T baixa do que em T alta, como esperado.")



## 5. Protocolo Completo de Simulação (Equilibração + Produção)

Assim como no notebook do Metropolis, esta função executa o protocolo da
Seção 4.5.3 do TCC, mas com a ressalva de escala temporal discutida acima:
aqui, `n_eq` e `n_prod` são contados em **passos de Wolff** (número de
aglomerados construídos e invertidos), não em MCS/site. Essa distinção será
formalizada com o tempo efetivo de autocorrelação na Seção 4.7.4 do TCC.

Os valores oficiais de produção usados no TCC (Seção 4.5.3) foram definidos
em MCS/site como unidade comum de referência entre os três algoritmos; aqui
usamos valores pequenos apenas para demonstração.


In [ ]:

def simular_wolff(
    L: int,
    T: float,
    n_eq: int,
    n_prod: int,
    intervalo_amostragem: int = 1,
    modo_inicial: str = "quente",
    rng: np.random.Generator = rng,
) -> dict:
    '''Executa o protocolo completo de simulação de Wolff para o modelo de
    Ising 2D (Seções 4.3.2 e 4.5.3 do TCC).

    Parameters
    ----------
    L : int
        Tamanho linear da rede.
    T : float
        Temperatura reduzida (k_B = 1).
    n_eq : int
        Número de passos de Wolff de equilibração (descartados).
    n_prod : int
        Número de passos de Wolff de produção (amostrados).
    intervalo_amostragem : int
        A cada quantos passos de Wolff uma medida é registrada.
    modo_inicial : {"fria", "quente"}
        Configuração inicial da rede.

    Returns
    -------
    dict
        Dicionário com "energia_por_sitio", "magnetizacao_abs_por_sitio",
        "tamanho_cluster" (n_cluster de cada passo de produção amostrado) e
        os parâmetros usados (L, T, n_eq, n_prod).
    '''
    N = L * L
    beta = 1.0 / T

    estado = inicializar_rede(L, modo=modo_inicial, rng=rng)
    vizinhos = construir_tabela_vizinhos(L)

    # --- Equilibração: descarta todas as configurações geradas ---
    for _ in range(n_eq):
        passo_wolff(estado, vizinhos, beta, rng)

    # --- Produção: amostra a cada `intervalo_amostragem` passos de Wolff ---
    energias = []
    magnetizacoes = []
    tamanhos_cluster = []
    for passo in range(n_prod):
        n_cluster = passo_wolff(estado, vizinhos, beta, rng)
        if passo % intervalo_amostragem == 0:
            e = energia_total(estado, vizinhos) / N
            m = abs(magnetizacao_total(estado)) / N
            energias.append(e)
            magnetizacoes.append(m)
            tamanhos_cluster.append(n_cluster)

    return {
        "energia_por_sitio": np.array(energias),
        "magnetizacao_abs_por_sitio": np.array(magnetizacoes),
        "tamanho_cluster": np.array(tamanhos_cluster),
        "L": L,
        "T": T,
        "n_eq": n_eq,
        "n_prod": n_prod,
    }


In [ ]:

# Teste rápido (parâmetros pequenos só para conferir que a função roda --
# NÃO é uma rodada de produção real). Note que, perto de Tc, poucos passos de
# Wolff já bastam para descorrelacionar, ao contrário do Metropolis.
resultado_teste = simular_wolff(L=32, T=2.269, n_eq=50, n_prod=300)
print("Energia média por sítio:", resultado_teste["energia_por_sitio"].mean())
print("Magnetização absoluta média por sítio:", resultado_teste["magnetizacao_abs_por_sitio"].mean())
print("Tamanho médio do aglomerado:", resultado_teste["tamanho_cluster"].mean())

plt.figure(figsize=(7, 3))
plt.plot(resultado_teste["energia_por_sitio"])
plt.xlabel("Amostra (1 por passo de Wolff)")
plt.ylabel(r"Energia por sítio $e$")
plt.title(f"Traço de energia -- L={resultado_teste['L']}, T={resultado_teste['T']}")
plt.tight_layout()
plt.show()



## 6. Testes de Sanidade Internos

Dois testes adicionais, específicos do Wolff, valem a pena conferir:

1. **Consistência de spin dentro do aglomerado**: por construção, todo sítio
   adicionado ao aglomerado tinha, no momento da adição, o mesmo spin da
   semente. Isso deve valer sempre, antes da inversão.
2. **Nenhum sítio duplicado**: o array `no_cluster` é booleano, então
   duplicidade não é fisicamente possível na nossa implementação -- mas
   vale confirmar que o tamanho do aglomerado bate com a soma do array.


In [ ]:

def teste_consistencia_cluster(L: int = 20, n_testes: int = 50, rng: np.random.Generator = rng) -> None:
    '''Confere que todo sitio pertencente a um aglomerado recem-construido
    compartilha o mesmo spin da semente (antes da inversao).
    '''
    vizinhos = construir_tabela_vizinhos(L)

    for _ in range(n_testes):
        estado = inicializar_rede(L, modo="quente", rng=rng)
        beta_aleatorio = rng.uniform(0.1, 1.0)
        p_add = probabilidade_ligacao(beta_aleatorio)

        no_cluster = construir_cluster(estado, vizinhos, p_add, rng)
        indices_cluster = np.where(no_cluster)[0]

        spins_no_cluster = estado[indices_cluster]
        assert len(set(spins_no_cluster.tolist())) <= 1, (
            "Aglomerado contem sitios com spins diferentes antes da inversao!"
        )

    print(f"OK: consistencia de spin verificada em {n_testes} aglomerados construidos.")

teste_consistencia_cluster()



## 7. Validação Preliminar contra a Solução Analítica de Onsager

Assim como no notebook do Metropolis, esta seção antecipa de forma rápida o
teste completo da Seção 4.4.1 do TCC. Como o Wolff descorrelaciona muito
mais rapidamente perto de $T_c$ (Seção 3.6), usamos aqui um número de passos
de produção bem menor do que seria necessário no Metropolis para uma
qualidade comparável -- mas ainda assim, **esta não é a validação oficial**,
que deve seguir os critérios estatísticos completos da Seção 4.4 e 4.7.


In [ ]:

# ATENÇÃO: valores pequenos só para demonstração -- rodar em poucos segundos.
# Para a validação oficial do TCC, usar os parâmetros da Seção 4.5 e comparar
# com barras de erro (método da Seção 4.7).

L_demo = 32
temperaturas_demo = np.array([1.5, 2.0, 2.269, 2.5, 3.0])

e_simulado = []
m_simulado = []
for T in temperaturas_demo:
    r = simular_wolff(L=L_demo, T=T, n_eq=50, n_prod=300)
    e_simulado.append(r["energia_por_sitio"].mean())
    m_simulado.append(r["magnetizacao_abs_por_sitio"].mean())

e_simulado = np.array(e_simulado)
m_simulado = np.array(m_simulado)

temperaturas_finas = np.linspace(1.2, 3.5, 200)
e_exato = np.array([energia_onsager(T) for T in temperaturas_finas])
m_exato = np.array([magnetizacao_onsager(T) for T in temperaturas_finas])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(temperaturas_finas, e_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax1.plot(temperaturas_demo, e_simulado, "o", color="teal", label=f"Wolff (L={L_demo}, demo rápida)")
ax1.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax1.set_xlabel("Temperatura $T$")
ax1.set_ylabel("Energia por sítio $e$")
ax1.legend()

ax2.plot(temperaturas_finas, m_exato, "-", color="black", label="Onsager (exato, $L\\to\\infty$)")
ax2.plot(temperaturas_demo, m_simulado, "o", color="teal", label=f"Wolff (L={L_demo}, demo rápida)")
ax2.axvline(temperatura_critica_onsager(), ls="--", color="gray", lw=1, label="$T_c$")
ax2.set_xlabel("Temperatura $T$")
ax2.set_ylabel("Magnetização absoluta por sítio $|m|$")
ax2.legend()

plt.suptitle("Validação preliminar (demonstração rápida -- não é a validação oficial do TCC)")
plt.tight_layout()
plt.show()



## Próximos Passos

- **Validação oficial (Seção 4.4 do TCC):** repetir a comparação acima com
  os parâmetros reais de produção e todas as redes $L \in \{16,32,64,128\}$,
  reportando incertezas estatísticas.
- **Swendsen-Wang:** implementar em notebook próprio (`swendsen_wang.ipynb`),
  reaproveitando `ising_utils.py` (inclusive `probabilidade_ligacao`, já
  usada aqui) e o algoritmo de Hoshen-Kopelman para identificação de
  múltiplos aglomerados simultâneos (Seção 4.3.3 do TCC).
- **Comparação de desempenho com Swendsen-Wang:** a discussão qualitativa
  sobre por que o Wolff tende a superar o Swendsen-Wang em CPU sequencial já
  foi apresentada na Seção 3.6 do TCC; a medição quantitativa real (tempo de
  execução, Fator de Mérito) será feita na Seção 4.8, após os três
  algoritmos estarem implementados e validados.
- **Tempo de autocorrelação efetivo do Wolff (Equação 4.47):** o cálculo de
  $\tau_{\text{efetivo, Wolff}}$ a partir de $\langle n_{\text{cluster}}
  \rangle$ (já medido neste notebook) será feito no notebook de análise de
  autocorrelação (Seção 4.7 do TCC), compartilhado entre os três algoritmos.
